In [ ]:
%load_ext autoreload
%autoreload 2

import functools
import itertools
from math import pi
import os


os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'


import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import ArrayLike, NDArray
import scipy
import tqdm.contrib.itertools
import xarray as xr

from fluxoniumcr import DATA_DIR
from fluxoniumcr.dressed_control_fluxonium import (
    create_driven_fluxonium,
    calculate_optimal_amplitude,
    calculate_polarization_and_error,
)
from fluxoniumcr.qubits.fluxonium import Fluxonium
from fluxoniumcr.qubits.product_basis import ProductBasis, DressedProductBasis

In [ ]:
drive_frequency_coord = xr.DataArray(
    np.linspace(0, 1.5, 151) * 2*pi,
    dims=['drive_frequency'],
    attrs=dict(
        long_name="Drive frequency",
        units="Grad/s",
    )
)
deltap_coord = xr.DataArray(
    np.linspace(0, 1, 11),
    dims=['deltap'],
)
EJ_coord = xr.DataArray(
    np.linspace(2, 8, 13) * 2*pi,
    dims=['EJ'],
    attrs=dict(
        units="Grad/s",
    )
)
EL_coord = xr.DataArray(
    np.linspace(0.3, 0.5, 11) * 2*pi,
    dims=['EL'],
    attrs=dict(
        units="Grad/s",
    )
)
EC_coord = xr.DataArray(
    np.array([1.2]) * 2*pi,
    dims=['EC'],
    attrs=dict(
        units="Grad/s",
    )
)
level_coord = xr.DataArray(
    np.arange(6),
    dims=['level'],
)

dataset = xr.Dataset()

dataset['drive_amplitude'] = xr.DataArray(
    np.nan,
    coords=[
        EC_coord,
        EJ_coord,
        EL_coord,
        drive_frequency_coord,
        deltap_coord,
    ]
)
dataset['drive_amplitude_base'] = xr.DataArray(
    np.nan,
    coords=[
        EC_coord,
        EJ_coord,
        EL_coord,
    ]
)
dataset['quasienergy'] = xr.DataArray(
    np.nan,
    coords=[
        EC_coord,
        EJ_coord,
        EL_coord,
        drive_frequency_coord,
        deltap_coord,
        level_coord,
    ]
)

dataset_path = DATA_DIR/"fluxonium_quasienergy_precomp"
dataset_name = "dataset"
def load_dataset(suffix=""):
    return xr.load_dataset(dataset_path/(dataset_name + suffix + ".hdf5"), engine='h5netcdf')
def save_dataset(dataset, suffix=""):
    dataset.to_netcdf(dataset_path/(dataset_name + suffix + ".hdf5"), engine='h5netcdf')

In [ ]:
from multiprocessing.pool import Pool
from fluxoniumcr.optimize import find_root, get_monotonic_increasing_intervals


def calculate_amp_for_deltap(
        fx,
        floquet_basis,
        deltap_data,
) -> NDArray[np.floating]:
    deltap_data = np.asarray(deltap_data)

    def fun(x: float) -> float:
        p0, p1, _ = calculate_polarization_and_error(
            fx,
            floquet_basis.query(x),
        )
        return abs(p0 - p1)
    def dfun(x: float) -> float:
        p0, p1, _ = calculate_polarization_and_error(
            fx,
            floquet_basis.query_perturbative(x, order=1),
        )
        y = p1[0] - p0[0]
        
        if abs(y) < 1e-4:
            dabsy = abs(p1[1] - p0[1])
        else:
            dabsy = np.sign(y)*(p1[1] - p0[1])

        return dabsy
    
    intervals, ranges = get_monotonic_increasing_intervals(
        fun,
        dfun,
        x0=floquet_basis.domain[0],
        x1=floquet_basis.domain[1],
        step_size=0.05,
        xtol=1e-9,
    )
    
    amp_data: list[float] = []
    for deltap in deltap_data:
        if deltap == 0.0:
            amp_data.append(0)
            continue
        
        for i, r in enumerate(ranges):
            if deltap >= r[0] and deltap <= r[1]:
                break
        else:
            amp_data.append(np.nan)
            continue
            
        x = scipy.optimize.bisect(
            lambda x: fun(x) - deltap,
            a=intervals[i][0],
            b=intervals[i][1],
        )
        amp_data.append(x)
    
    return np.array(amp_data, dtype=float)


def doit(fx, deltap_data, drive_freq):
    fx_evals = fx.eigenvalues
    fx_freq10 = fx_evals[1] - fx_evals[0]
    n_op = fx.get_operator('charge')
    Ω0 = fx_freq10/abs(n_op[0, 1])
    amps_lookup = np.linspace(0, 1, 51) * Ω0

    if drive_freq < fx_freq10:
        return (
            np.full(len(deltap_data), np.nan),
            np.full((len(deltap_data), fx.dim), np.nan),
        )
    
    floquet_basis = create_driven_fluxonium(
        fx,
        drive_freq,
        N=128,
        glide_reflection=True,
    )
    floquet_basis.generate_lookup(amps_lookup)
    
    amp_data = calculate_amp_for_deltap(fx, floquet_basis, deltap_data)
    
    quasienergy_data = np.array([
        floquet_basis.quasienergies(p)
        if np.isfinite(p) else np.full(fx.dim, np.nan)
        for p in amp_data
    ])
    
    # Shift odd quasienergies back up by +drive_freq.
    quasienergy_data[:, 1::2] += drive_freq
    
    return amp_data, quasienergy_data

In [ ]:
tqdm_total = len(dataset.EJ) * len(dataset.EC) * len(dataset.EL) * len(dataset.drive_frequency)

with tqdm.tqdm(total=tqdm_total) as pbar:
    for EJ, EC, EL in itertools.product(
            dataset.EJ.data,
            dataset.EC.data,
            dataset.EL.data,
    ):
        ds_outer = dataset.loc[dict(
            EJ=EJ,
            EC=EC,
            EL=EL,
        )]

        fx = Fluxonium(
            EJ=EJ,
            EC=EC,
            EL=EL,
            flux=0.5,
            cutoff=128,
            dim=8,
        )
        fx_evals = fx.eigenvalues
        fx_freq10 = fx_evals[1] - fx_evals[0]
        n_op = fx.get_operator('charge')
        Ω0 = fx_freq10/abs(n_op[0, 1])

        ds_outer.drive_amplitude_base[()] = Ω0

        amps_lookup = np.linspace(0, 1, 51) * Ω0

        with Pool(processes=10) as pool:
            for drive_freq, result in zip(
                    ds_outer.drive_frequency.data,
                    pool.imap(
                        functools.partial(doit, fx, dataset.deltap.data),
                        ds_outer.drive_frequency.data,
                    ),
                    
            ):
                ds = ds_outer.loc[dict(drive_frequency=drive_freq)]
                
                amp_data, quasienergy_data = result
                
                ds.drive_amplitude[:] = amp_data
                ds.quasienergy[:] = quasienergy_data[:, :len(ds.level)]

                pbar.update(1)

In [ ]:
save_dataset(dataset)